# Fix 01 — align model comparison with the review decision

This notebook compares the original baseline, logistic regression, and Random Forest.
The **primary metric is precision@50**. Learned-model selection uses only an inner
client validation split; all methods are then evaluated on the same outer client test set.
Exact selection ties prefer logistic regression. Canonical page-key ordering resolves ranking ties;
tie sensitivity is also reported.

**Stage 1 only:** the old February/March label is deliberately preserved to isolate this repair.
Unequal windows, zero-click eligibility, baseline validity and chronological validation remain pending.
The outer clients have been examined before, so this is exploratory, not a new sealed confirmation.
The baseline is reported as a comparator; no model is recommended for production at this stage.

Run in a fresh Colab runtime with approved warehouse access and `HF_TOKEN` in Secrets.
Never paste the token into code/chat. Run all cells. Share only aggregate JSON and the figure.
Do not replace the old paper's numbers until the warehouse run and later repairs are complete.


In [ ]:
import os, sys, subprocess
SMOKE = os.environ.get('FLYRANK_FIX01_SMOKE') == '1'
if not SMOKE:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'numpy==2.2.6', 'pandas==2.2.3', 'scipy==1.16.3',
        'scikit-learn==1.7.2', 'matplotlib==3.10.7',
        'duckdb==1.4.1', 'huggingface_hub==1.0.1'])
print('SYNTHETIC SOFTWARE CHECK ONLY' if SMOKE else 'Warehouse mode: approved access required')


In [ ]:
"""Stage 1: compare all candidates; select only within the outer training clients.

The original February/March outcome is deliberately retained by the calling notebook.
Results remain exploratory until the separate outcome and temporal-validation repairs.
"""
from __future__ import annotations

import hashlib
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ['impressions_prior30', 'clicks_prior30', 'ctr_prior30', 'avg_position_prior30']
CATEGORICAL = ['content_type', 'main_intent']
FEATURES = NUMERIC + CATEGORICAL
KEYS = ['client_hash_id', 'content_hash_id']
TARGET = 'decoupling_signature'
CANDIDATES = ['Logistic Regression', 'Random Forest']
PRIMARY_K = 50


def make_model(name, seed=42, n_trees=300):
    # Each pipeline owns its transformer. Fitting one candidate cannot mutate another.
    prep = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ])
    if name == 'Logistic Regression':
        clf = LogisticRegression(max_iter=1000, random_state=seed)
    elif name == 'Random Forest':
        clf = RandomForestClassifier(n_estimators=n_trees, random_state=seed, n_jobs=-1)
    else:
        raise ValueError(f'Unknown candidate: {name}')
    return Pipeline([('prep', prep), ('clf', clf)])


def rank_metrics(scores, labels, ks=(25, 50, 100, 200)):
    scores, labels = np.asarray(scores, dtype=float), np.asarray(labels, dtype=int)
    if len(scores) != len(labels) or not len(scores):
        raise ValueError('Scores and labels must be nonempty and aligned.')
    if not np.isfinite(scores).all() or set(np.unique(labels)) != {0, 1}:
        raise ValueError('Finite scores and both binary outcome classes are required.')
    if max(ks) > len(labels):
        raise ValueError('Not enough evaluation rows for every declared K.')
    # Caller canonically orders page keys. Stable ties follow those keys, never the label.
    order = np.argsort(-scores, kind='stable')
    prevalence = float(labels.mean())
    out = {'n_eval': len(labels), 'positive_count': int(labels.sum()),
           'base_rate': prevalence, 'roc_auc': float(roc_auc_score(labels, scores)),
           'average_precision': float(average_precision_score(labels, scores)),
           'unique_score_count': int(np.unique(scores).size),
           'zero_score_fraction': float((scores == 0).mean())}
    for k in ks:
        hits = int(labels[order[:k]].sum())
        cutoff = scores[order[k-1]]
        above = int((scores > cutoff).sum())
        at = scores == cutoff
        remaining = k-above
        tied_hits = int(labels[at].sum())
        tied_count = int(at.sum())
        above_hits = int(labels[scores > cutoff].sum())
        out.update({f'hits@{k}': hits, f'precision@{k}': hits/k,
                    f'recall@{k}': hits/int(labels.sum()),
                    f'lift@{k}': hits/k/prevalence,
                    f'cutoff_tie_count@{k}': tied_count,
                    # Expected performance under uniformly random cutoff tie breaking.
                    f'tie_expected_precision@{k}':
                        (above_hits + remaining*tied_hits/tied_count)/k,
                    f'tie_min_precision@{k}':
                        (above_hits + max(0, remaining-(tied_count-tied_hits)))/k,
                    f'tie_max_precision@{k}':
                        (above_hits + min(remaining, tied_hits))/k})
    return out


def choose_candidate(validation_table, primary_k=PRIMARY_K):
    """Fixed simplicity preference on exact ties; never use the outer test table."""
    learned = validation_table.set_index('method').loc[CANDIDATES]
    scores = learned[f'precision@{primary_k}']
    return next(name for name in CANDIDATES if scores[name] == scores.max())


def fit_reference(train, evaluation):
    """Original median-gap rule, fitted on training rows; diagnostics expose collapse."""
    bins = [0, 3, 10, 20, 50, np.inf]
    train_bucket = pd.cut(train.avg_position_prior30, bins, include_lowest=True)
    eval_bucket = pd.cut(evaluation.avg_position_prior30, bins, include_lowest=True)
    medians = train.groupby(train_bucket, observed=False).ctr_prior30.median()
    reference = eval_bucket.map(medians).astype(float)
    if reference.isna().any():
        raise ValueError('Baseline has an unsupported/missing position bucket; audit it.')
    visible = evaluation.impressions_prior30 >= train.impressions_prior30.median()
    return (visible * (reference-evaluation.ctr_prior30).clip(lower=0)).to_numpy()


def compare_models(frame, seed=42, n_trees=300, ks=(25, 50, 100, 200)):
    if PRIMARY_K not in ks:
        raise ValueError('The prespecified primary K=50 must be evaluated.')
    d = frame.sort_values(KEYS, kind='stable').reset_index(drop=True).copy()
    if d.duplicated(KEYS).any():
        raise ValueError('Page keys are not unique; audit the joins before fitting.')
    required = FEATURES + KEYS + [TARGET]
    if d[required].isna().any().any():
        raise ValueError('Missing required values; retain a documented cohort policy.')
    outer = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed)
    outer_train_idx, test_idx = next(outer.split(d, groups=d.client_hash_id))
    outer_train, test = d.iloc[outer_train_idx].copy(), d.iloc[test_idx].copy()
    inner = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed+1)
    inner_fit_idx, val_idx = next(inner.split(outer_train, groups=outer_train.client_hash_id))
    fit, validation = outer_train.iloc[inner_fit_idx], outer_train.iloc[val_idx]
    groups = [set(x.client_hash_id) for x in (fit, validation, test)]
    if any(groups[i] & groups[j] for i in range(3) for j in range(i+1, 3)):
        raise AssertionError('Client overlap between fit, validation, and test.')
    for part in (fit, validation, test):
        if part[TARGET].nunique() != 2 or len(part) < max(ks):
            raise ValueError('Each split requires both classes and enough rows for K.')

    def evaluate(train, evaluation):
        models, scores = {}, {'Baseline rule': fit_reference(train, evaluation)}
        for name in CANDIDATES:
            model = make_model(name, seed=seed, n_trees=n_trees)
            model.fit(train[FEATURES], train[TARGET])
            models[name] = model
            scores[name] = model.predict_proba(evaluation[FEATURES])[:, 1]
        rows = [{'method': name, **rank_metrics(s, evaluation[TARGET], ks)}
                for name, s in scores.items()]
        return pd.DataFrame(rows), models, scores

    validation_table, _, _ = evaluate(fit, validation)
    selected = choose_candidate(validation_table)
    # Freeze selected before any outer test metrics are calculated.
    test_table, fitted, test_scores = evaluate(outer_train, test)
    membership = d[KEYS].copy()
    membership['split'] = 'inner_fit'
    membership.loc[outer_train.index[val_idx], 'split'] = 'inner_validation'
    membership.loc[test.index, 'split'] = 'outer_test'
    cohort_hash = hashlib.sha256(pd.util.hash_pandas_object(
        d[KEYS+FEATURES+[TARGET]], index=False).values.tobytes()).hexdigest()
    manifest = {
        'status': 'exploratory_stage_1_not_publication_validation',
        'primary_metric': 'precision@50',
        'selection_data': 'inner client validation only',
        'selection_tie_rule': 'prefer Logistic Regression',
        'ranking_tie_rule': 'canonical client/content key order, stable sort',
        'selected_method': selected, 'seed': seed, 'inner_seed': seed+1,
        'n_trees': n_trees, 'cohort_sha256': cohort_hash,
        'n_rows': len(d), 'n_clients': int(d.client_hash_id.nunique()),
        'partitions': {name: {'n_rows':len(part),
                             'n_clients':int(part.client_hash_id.nunique()),
                             'base_rate':float(part[TARGET].mean())}
                       for name,part in [('inner_fit',fit),('inner_validation',validation),
                                         ('outer_train',outer_train),('outer_test',test)]},
        'environment': {'python':platform.python_version(), 'numpy':np.__version__,
                        'pandas':pd.__version__, 'scikit_learn':sklearn.__version__},
        'known_pending_fixes': ['unequal calendar windows', 'zero-click eligibility',
                               'baseline validity', 'time-forward validation',
                               'client/time-aware uncertainty', 'prospective queue utility'],
        'note': 'Outer clients were examined in prior work; do not describe this as a new sealed test.'
    }
    return {'frame':d, 'validation':validation_table, 'test':test_table,
            'selected_method':selected, 'manifest':manifest, 'membership':membership,
            'test_frame':test, 'test_scores':test_scores, 'models':fitted}


In [ ]:
import re
import json
from pathlib import Path

if SMOKE:
    rng = np.random.default_rng(19)
    n = 4800
    impressions = rng.integers(100, 10000, n)
    clicks = rng.integers(1, 100, n)
    position = rng.uniform(1, 70, n)
    df = pd.DataFrame({
        'client_hash_id':np.repeat([f'synthetic_client_{i:02d}' for i in range(16)], 300),
        'content_hash_id':[f'synthetic_page_{i:05d}' for i in range(n)],
        'impressions_prior30':impressions, 'clicks_prior30':clicks,
        'ctr_prior30':100*clicks/impressions, 'avg_position_prior30':position,
        'content_type':rng.choice(['type_a','type_b'], n),
        'main_intent':rng.choice(['informational','commercial'], n),
        'decoupling_signature':rng.binomial(1, .04+.18*(position < 15), n),
    })
else:
    import duckdb
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Set HF_TOKEN in Colab Secrets or your local environment; do not paste it into this notebook or chat.')
    if not re.fullmatch(r'hf_[A-Za-z0-9]+', token):
        raise RuntimeError('Expected a Hugging Face read token in the approved secret location.')
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")
    del token
    REL = 'hf://datasets/FlyRank/internship-warehouse'
    FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
    MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
    feb_page = con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30,
               SUM(gsc_clicks) AS clicks_prior30,
               CASE WHEN SUM(gsc_impressions)>0
                    THEN SUM(gsc_sum_position)/SUM(gsc_impressions) ELSE NULL END AS avg_position_prior30
        FROM {FEB} WHERE gsc_data_available IS TRUE
        GROUP BY 1,2 HAVING SUM(gsc_impressions)>0
    """).df()
    feb_page['ctr_prior30'] = 100*feb_page.clicks_prior30/feb_page.impressions_prior30
    content_meta = con.sql(f"""
        SELECT content_hash_id, content_type, main_intent
        FROM read_parquet('{REL}/dim_content.parquet')
    """).df()
    # Assert the existing content-only key assumption instead of silently duplicating pages.
    feb_page = feb_page.merge(content_meta, on='content_hash_id', how='left', validate='many_to_one')
    mar_page = con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_last30,
               SUM(gsc_clicks) AS clicks_last30
        FROM {MAR} WHERE gsc_data_available IS TRUE GROUP BY 1,2
    """).df()
    df = feb_page.merge(mar_page, on=KEYS, how='inner', validate='one_to_one')
    df['impr_change_pct'] = 100*(df.impressions_last30-df.impressions_prior30)/df.impressions_prior30
    df['click_change_pct'] = 100*(df.clicks_last30-df.clicks_prior30)/df.clicks_prior30.replace(0,np.nan)
    df[TARGET] = (df.impr_change_pct.between(-10,10) & (df.click_change_pct<=-15)).astype(int)
    df = df.dropna(subset=['avg_position_prior30','content_type','main_intent'])
    con.close()
    # Stage 1 deliberately preserves the historical label; repair calendar/eligibility next.

print({'mode':'synthetic' if SMOKE else 'warehouse', 'n_pages':len(df),
       'n_clients':int(df.client_hash_id.nunique()), 'base_rate':float(df[TARGET].mean())})


In [ ]:
result = compare_models(df, n_trees=12 if SMOKE else 300)
columns = ['method','hits@50','precision@50','recall@50','average_precision','roc_auc','base_rate']
print('INNER VALIDATION — learned-candidate selection by precision@50')
display(result['validation'][columns])
print('Selected learned candidate:', result['selected_method'])
print('OUTER TEST — all methods, same rows; selection already frozen')
display(result['test'][columns])
display(result['test'][['method','unique_score_count','zero_score_fraction',
    'cutoff_tie_count@50','tie_min_precision@50','tie_expected_precision@50','tie_max_precision@50']])
print('No production recommendation is made in Stage 1. Calendar, eligibility, baseline and temporal repairs remain.')


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

out = Path('work/outputs/fix01_smoke' if SMOKE else 'work/outputs/fix01')
out.mkdir(parents=True, exist_ok=True)
manifest = result['manifest']
manifest['mode'] = 'synthetic_software_check' if SMOKE else 'warehouse_exploratory'
manifest['historical_target_preserved'] = True
payload = {'manifest':manifest,
    'validation':result['validation'].to_dict(orient='records'),
    'outer_test':result['test'].to_dict(orient='records')}
with (out/'model_comparison.json').open('w',encoding='utf-8') as f:
    json.dump(payload,f,indent=2,allow_nan=False)

fig, axes = plt.subplots(1,3,figsize=(15,4))
for name,scores in result['test_scores'].items():
    labels = result['test_frame'][TARGET]
    fpr,tpr,_ = roc_curve(labels,scores)
    precision,recall,_ = precision_recall_curve(labels,scores)
    axes[0].plot(fpr,tpr,label=name)
    axes[1].plot(recall,precision,label=name)
    row = result['test'].set_index('method').loc[name]
    axes[2].plot([25,50,100,200],[row[f'precision@{k}'] for k in [25,50,100,200]],marker='o',label=name)
axes[0].plot([0,1],[0,1],'k--',alpha=.4)
axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='ROC — all methods')
for ax in axes[1:]:
    ax.axhline(result['test_frame'][TARGET].mean(),color='grey',linestyle='--')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Precision–recall — all methods')
axes[2].set(xlabel='Review budget K',ylabel='Precision@K',title='Review-budget performance')
for ax in axes: ax.legend(fontsize=7)
fig.suptitle('SYNTHETIC SOFTWARE CHECK — NOT RESEARCH RESULTS' if SMOKE else 'Stage 1 exploratory comparison — outcome/time repairs pending')
fig.tight_layout()
fig.savefig(out/'model_comparison.png',dpi=150)
plt.show()
print('Saved aggregate metrics and figure to',out)
print('No tokens, page-level data, or client identifiers are exported.')


## Interpretation checklist

- Which method leads on precision@50? Which leads on AUC? Report both.
- Does the selected learned candidate beat a functioning baseline by a useful margin?
- Do many tied/zero baseline scores make its top-K result arbitrary?
- Are the results stable across reasonable review budgets?
- These rows are correlated within clients and all outcomes come from one month pair.
  No independence-based confidence claim or causal/business recovery claim is made here.

Next repair: equal-duration observation/outcome windows and an explicit zero-click eligibility policy.
Historical Week 5 output (not this run) was logistic regression .24 P@50 versus forest .08,
with AUC .709 versus .810. Those values are not copied into any new result file.
